# Compound Wind-Hydro Energy Droughts in Patagonia


### Research Question
**Can compound wind hydro energy droughts in Patagonia be predicted from 
large scale climate modes, and if so, how far in advance?**

### Project Overview

This analysis integrates 47 years of climate data (1979–2025) with regional wind and hydroelectric generation potential across Patagonia 
(38°S–47.5°S, 72°W–62°W) to determine whether the Southern Annular Mode (SAM), Oceanic Niño Index (ONI), and Indian Ocean Dipole (IOD) carry predictive skill for compound energy droughts.

**Key regions of focus:**
- **Hydroelectric:** Limay-Neuquén basin dams (Chocón, Piedra del Águila, Alicurá, Futaleufú)
- **Wind:** Atlantic coast wind farms (Rawson, Trelew, Puerto Madryn, Manantiales Behr)
- **Temporal domain:** Monthly aggregation, 1979–2025 (47 years)
- **Compound metric:** WHDI = standardised wind anomaly + standardised runoff anomaly

### Summary of Findings

**Yes, compound droughts are predictable** but the predictive source inverts across the forecast horizon.

**Local conditions dominate short leads (1–2 months):** +68% and +39% skill vs climatology

**Climate modes dominate medium leads (5–6 months):** +4.6% skill, p<0.05 (IOD-driven)

**Lead 3 is a transition cliff edge:** Both sources collapse, forecast skill near zero

**Super El Niño events are hardest to predict:** 1997–98 and 2015–16 show 1.6× larger forecast errors despite extreme climate signals

**Operational implication:** Forecast systems must switch paradigms, use local persistence for months 1–2, climate teleconnections for months 5–6, with no 
single hybrid model dominating both horizons.

In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import pickle

warnings.filterwarnings("ignore")

# Set style for all visualizations in this notebook
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9
plt.rcParams["legend.fontsize"] = 9
plt.rcParams["lines.linewidth"] = 1.5
plt.rcParams["patch.linewidth"] = 0.5

# Notebook 2
df_whdi = pd.read_csv(
    "../data/processed/whdi_timeseries.csv", index_col="date", parse_dates=True
)
print(f"WHDI Timeseries: {df_whdi.shape}")

df_droughts = pd.read_csv("../results/tables/drought_catalog.csv")
print(f"Drought Catalog: {df_droughts.shape}")

df_climatology = pd.read_csv("../results/tables/monthly_climatology.csv", index_col=0)
print(f"Monthly Climatology: {df_climatology.shape}")

# Notebook 3
df_lag_corr = pd.read_csv("../results/tables/lag_correlations.csv", index_col=0)
print(f"Lag Correlations: {df_lag_corr.shape}")

df_seasonal_lag = pd.read_csv(
    "../results/tables/seasonal_lag_correlations.csv", index_col=0
)
print(f"Seasonal Lag Correlations: {df_seasonal_lag.shape}")

df_trends = pd.read_csv("../results/tables/trend_results.csv")
print(f"Trend Results: {df_trends.shape}")

# Notebook 4
df_performance = pd.read_csv("../results/tables/model_performance.csv")
print(f"Model Performances: {df_performance.shape}")

df_skills = pd.read_csv("../results/tables/skill_scores.csv")
print(f"Skill Scores: {df_skills.shape}")

df_significance = pd.read_csv("../results/tables/significance_tests.csv")
print(f"Significance Tests: {df_significance.shape}")

df_anomalies = pd.read_csv("../results/tables/anomalous_months_summary.csv")
print(f"Anomalous Months: {df_anomalies.shape}")

with open("../results/tables/feature_decisions.json", "r") as f:
    feature_decisions = json.load(f)
print(f"Feature decisions: {len(feature_decisions)} parameters")

with open("../results/tables/coef_series.pkl", "rb") as f:
    coef_series = pickle.load(f)
print(f"ElasticNet coefficients: {len(coef_series)} leads")

WHDI Timeseries: (564, 13)
Drought Catalog: (31, 7)
Monthly Climatology: (12, 16)
Lag Correlations: (195, 8)
Seasonal Lag Correlations: (468, 7)
Trend Results: (5, 6)
Model Performances: (3456, 9)
Skill Scores: (2304, 8)
Significance Tests: (288, 11)
Anomalous Months: (28, 7)
Feature decisions: 4 parameters
ElasticNet coefficients: 12 leads


In [35]:
# Study Domain
STUDY_BOUNDS = [-72.0, -47.5, -62.0, -38.0]
STUDY_CENTRE = [-42.5, -67.5]

# Drought Thresholds
DROUGHT_THRESHOLDS = [-1.0, -1.5, -2.0]

# Lead Times Analysed
LEAD_TIMES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
PRIMARY_TARGET = "whdi_3"
SECONDARY_TARGET = "whdi_6"

# Models and ML things
MODELS = [
    "elasticnet",
    "random_forest",
    "xgboost",
    "lstm",
    "climatology",
    "persistence",
]
ML_MODELS = ["elasticnet", "random_forest", "xgboost", "lstm"]
FEATURE_GROUPS = ["climate_only", "local_only", "combined"]
CLIMATE_MODES = ["SAM", "ONI", "IOD"]

# Maintain consistency across all plots etc
COLOURS = {
    "elasticnet": "#0095ff",
    "random_forest": "#ff0000",
    "xgboost": "#ff9900",
    "lstm": "#00ff2a",
    "climatology": "black",
    "persistence": "lightgrey",
    "climate_only": "#E91E63",
    "local_only": "#2196F3",
    "combined": "#4CAF50",
}

print("Parameters set:")
print(f"  Study region: {STUDY_BOUNDS}")
print(f"  Lead times: {LEAD_TIMES}")
print(f"  Primary target: {PRIMARY_TARGET}")
print(f"  Feature groups: {FEATURE_GROUPS}")
print(f"  ML models: {ML_MODELS}")

Parameters set:
  Study region: [-72.0, -47.5, -62.0, -38.0]
  Lead times: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  Primary target: whdi_3
  Feature groups: ['climate_only', 'local_only', 'combined']
  ML models: ['elasticnet', 'random_forest', 'xgboost', 'lstm']


## Utility Functions

In [36]:
def get_best_model_at_lead(df_skills_subset, target, lead, feature_group):
    """
    Return the best performing model (by skill vs climatology) at a given lead.
    """
    subset = df_skills_subset[
        (df_skills_subset["target"] == target)
        & (df_skills_subset["lead"] == lead)
        & (df_skills_subset["feature_group"] == feature_group)
    ]
    if len(subset) == 0:
        return None
    return subset.groupby("model")["skill_vs_clim"].mean().idxmax()


def summarise_skill_at_lead(df_skills_subset, target, lead):
    """
    Summarise skill across feature groups at a single lead time.
    Returns dict with climate_only, local_only, combined mean skills.
    """
    best_model = get_best_model_at_lead(df_skills_subset, target, lead, "combined")
    if best_model is None:
        return None

    result = {}
    for group in FEATURE_GROUPS:
        skill = df_skills_subset[
            (df_skills_subset["target"] == target)
            & (df_skills_subset["lead"] == lead)
            & (df_skills_subset["feature_group"] == group)
            & (df_skills_subset["model"] == best_model)
        ]["skill_vs_clim"].mean()
        result[group] = skill

    return result


def count_drought_events(whdi_ts, threshold=DROUGHT_THRESHOLDS[0]):
    """
    Count contiguous drought events in WHDI timeseries.
    Returns list of (start_date, end_date, duration, peak_severity).
    """
    in_drought = whdi_ts < threshold
    events = []

    start = None
    for date, is_drought in in_drought.items():
        if is_drought and start is None:
            start = date
        elif not is_drought and start is not None:
            end = date
            duration = (end - start).days // 30
            peak = whdi_ts[start:end].min()
            events.append((start, end, duration, peak))
            start = None

    return events

## Notebook Structure

This notebook is organised into 7 sections:

1. **SETUP & CONTEXT**: Load data, define parameters, summary

2. **EXECUTIVE SUMMARY DASHBOARD**: Key numbers at a glance

3. **INTERACTIVE MAP**: Spatial context of study region & infrastructure

4. **RESULTS COMPILATION TABLE**: All key metrics in one place

5. **SYNTHESIS NARRATIVE**: Full interpretation of findings

6. **KEY FIGURES**: Polished, publication-ready visualizations

7. **RESEARCH QUESTION ANSWER**: Direct response with caveats

# SECTION 2: EXECUTIVE SUMMARY DASHBOARD

## Key Findings in one figure

The dashboard below summarises the critical results from all four analysis notebooks in a single 2×3 visualization. More detailed results follow in subsequent sections.

In [ ]:
print(" " * 26 + "Compiling Metrics for Dashboard")
print("=" * 81)

# Drought events
print("Longest Drought Event:")
if len(df_droughts) > 0:
    longest = df_droughts.loc[df_droughts["duration"].idxmax()]
    longest_duration = int(longest["duration"])
    longest_start = longest["start"]
    longest_end = longest["end"]
    longest_peak = float(longest["peak_severity"])

    print(f"   Duration: {longest_duration} months")
    print(f"   Period: {longest_start} to {longest_end}")
    print(f"   Peak severity: {longest_peak:.2f} (σ)")
else:
    longest_duration = 0
    longest_peak = 0

# Best forecasting skills
print("\nBest Forecast Skill (Leads 1 and 2):")

# Lead 1 and 2 with local only as its greatest peak
lead1_local = df_skills[
    (df_skills["target"] == PRIMARY_TARGET)
    & (df_skills["lead"] == 1)
    & (df_skills["feature_group"] == "local_only")
    & (df_skills["model"] == "elasticnet")
]["skill_vs_clim"].mean()

lead2_local = df_skills[
    (df_skills["target"] == PRIMARY_TARGET)
    & (df_skills["lead"] == 2)
    & (df_skills["feature_group"] == "local_only")
    & (df_skills["model"] == "elasticnet")
]["skill_vs_clim"].mean()

best_lead_skill = max(lead1_local, lead2_local)
best_lead = 1 if lead1_local > lead2_local else 2

print(f"   Lead time: {best_lead} month(s)")
print(f"   Skill vs climatology: {best_lead_skill:.1%}")
print("   Model: ElasticNet (Local only features)")

# Worst forecasting skill
print("\nForecast Transition Zone Skill (Lead 3):")
lead3_all = df_skills[
    (df_skills["target"] == PRIMARY_TARGET)
    & (df_skills["lead"] == 3)
    & (df_skills["feature_group"] == "combined")
    & (df_skills["model"] == "elasticnet")
]["skill_vs_clim"].mean()

print("   Lead time: 3 months")
print(f"   Skill vs climatology: {lead3_all:.1%}")
print("   Model: ElasticNet (Combined features, neither feature group dominates)")

# Best climate mode signal
print("\nClimate Mode Best Forecast (Leads 5 and 6):")

lead5_climate = df_skills[
    (df_skills["target"] == PRIMARY_TARGET)
    & (df_skills["lead"] == 5)
    & (df_skills["feature_group"] == "climate_only")
    & (df_skills["model"] == "elasticnet")
]["skill_vs_clim"].mean()

lead6_climate = df_skills[
    (df_skills["target"] == PRIMARY_TARGET)
    & (df_skills["lead"] == 6)
    & (df_skills["feature_group"] == "climate_only")
    & (df_skills["model"] == "elasticnet")
]["skill_vs_clim"].mean()

lead6_sig = df_significance[
    (df_skills["target"] == PRIMARY_TARGET)
    & (df_skills["lead"] == 6)
    & (df_skills["feature_group"] == "climate_only")
    & (df_skills["model"] == "elasticnet")
]["p_value"].values

sig_marker = (
    "significant p<0.05"
    if len(lead6_sig) > 0 and lead6_sig[0] < 0.05
    else "not significant"
)

print(f"   Lead 5: {lead5_climate:.1%} skill")
print(f"   Lead 6: {lead6_climate:.1%} skill ({sig_marker})")
print("   Driver: IOD lags 2-4, 8-11 (confirmed by feature survival)")

# Best Model Architecture
print("\nBest Model Architecture:")

elasticnet_mean = df_skills[
    (df_skills["target"] == PRIMARY_TARGET)
    & (df_skills["feature_group"] == "combined")
    & (df_skills["model"] == "elasticnet")
]["skill_vs_clim"].mean()

rf_mean = df_skills[
    (df_skills["target"] == PRIMARY_TARGET)
    & (df_skills["feature_group"] == "combined")
    & (df_skills["model"] == "random_forest")
]["skill_vs_clim"].mean()

xgboost_mean = df_skills[
    (df_skills["target"] == PRIMARY_TARGET)
    & (df_skills["feature_group"] == "combined")
    & (df_skills["model"] == "xgboost")
]["skill_vs_clim"].mean()

best_model_name = "ElasticNet"
best_model_skill = elasticnet_mean

print(f"  Winner: {best_model_name}")
print(f"  Mean skill (combined): {best_model_skill:.1%}")
print(f"  vs Random Forest: {(elasticnet_mean - rf_mean):.1%}")
print(f"  vs XGBoost: {(elasticnet_mean - xgboost_mean):.1%}")
print("  Reason: L1/L2 regularisation handles spare teleconnection structure")

# Climate mode heirarchy
print("\nClimaate mode importance ranking:")

# Count feature survival at leads 5–6
iod_survives = sum(
    [
        ("iod_lag" in f) and (coef_series[lead][f] != 0)
        for lead in [5, 6]
        for f in coef_series[lead].index
        if "iod_lag" in f
    ]
)
sam_survives = sum(
    [
        ("sam_lag" in f) and (coef_series[lead][f] != 0)
        for lead in [5, 6]
        for f in coef_series[lead].index
        if "sam_lag" in f
    ]
)
oni_survives = sum(
    [
        ("oni_lag" in f) and (coef_series[lead][f] != 0)
        for lead in [5, 6]
        for f in coef_series[lead].index
        if "oni_lag" in f
    ]
)

print("  IOD (Indian Ocean Dipole)")
print(f"    Features surviving at leads 5-6: {iod_survives} lags")
print("    Coefficient magnitude: Large (dark blue in heatmap)")
print("    Mechanism: Moisture transport to Andes")
print("  ONI (Oceanic Niño Index)")
print(f"    Features surviving: {oni_survives} lags")
print("    Appears at longer leads (9+)")
print("  SAM (Southern Annular Mode)")
print(f"    Features surviving: {sam_survives} lags")
print("    Verdict: Statistically significant but weak predictive utility")

dashboard_data = {
    "longest_duration": longest_duration,
    "longest_peak": longest_peak,
    "best_lead": best_lead,
    "best_skill": best_lead_skill,
    "lead3_skill": lead3_all,
    "lead6_skill": lead6_climate,
    "best_model": best_model_name,
    "iod_survives": iod_survives,
    "sam_survives": sam_survives,
    "oni_survives": oni_survives,
}

                          Compiling Metrics for Dashboard
Longest Drought Event:
   Duration: 11 months
   Period: 2016-02-01 to 2017-01-01
   Peak severity: -3.25 (σ)

Best Forecast Skill (Leads 1 and 2):
   Lead time: 1 month(s)
   Skill vs climatology: 68.0%
   Model: ElasticNet (Local only features)

Forecast Transition Zone Skill (Lead 3):
   Lead time: 3 months
   Skill vs climatology: 3.7%
   Model: ElasticNet (Combined features, neither feature group dominates)

Climate Mode Best Forecast (Leads 5 and 6):
   Lead 5: 3.9% skill
   Lead 6: 4.6% skill (not significant)
   Driver: IOD lags 2-4, 8-11 (confirmed by feature survival)

Best Model Architecture:
  Winner: ElasticNet
  Mean skill (combined): 10.9%
  vs Random Forest: 2.1%
  vs XGBoost: 0.2%
  Reason: L1/L2 regularisation handles spare teleconnection structure

Climaate mode importance ranking:
  IOD (Indian Ocean Dipole)
    Features surviving at leads 5-6: 11 lags
    Coefficient magnitude: Large (dark blue in heatmap)
 